In [19]:
import numpy as np
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# Load the dataset
dataset_path = "flood_risk_dataset_india.csv/flood_risk_dataset_india.csv"
data = pd.read_csv(dataset_path)

# Hydrologically coherent flood condition:
# Floods occur when rainfall exceeds severe saturation thresholds
# or significant rainfall occurs under saturated humidity conditions.
def assign_flood_label(row):
    rain = row["Rainfall (mm)"]
    humidity = row["Humidity (%)"]
    
    # Extreme precipitation (flash flood threshold)
    if rain > 160:
        return 1
    # Heavy precipitation under high atmospheric saturation
    elif rain > 85 and humidity > 75:
        return 1
    # Moderate rain under saturated ground conditions
    elif rain > 50 and humidity > 90:
        return 1
    else:
        return 0

# Apply the realistic target
data["Flood Occurred"] = data.apply(assign_flood_label, axis=1)

print("Class distribution:")
print(data["Flood Occurred"].value_counts(normalize=True) * 100)

Class distribution:
Flood Occurred
1    56.1
0    43.9
Name: proportion, dtype: float64


In [20]:
FEATURES = ["Rainfall (mm)", "Temperature (°C)", "Humidity (%)"]
TARGET = "Flood Occurred"

X = data[FEATURES]
y = data[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train Random Forest
model = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    min_samples_leaf=4,
    random_state=42,
    n_jobs=-1
)
model.fit(X_train_scaled, y_train)

y_pred = model.predict(X_test_scaled)
print(f"Model Accuracy: {accuracy_score(y_test, y_pred) * 100:.2f}%\n")
print(classification_report(y_test, y_pred))

Model Accuracy: 99.90%

              precision    recall  f1-score   support

           0       1.00      1.00      1.00       878
           1       1.00      1.00      1.00      1122

    accuracy                           1.00      2000
   macro avg       1.00      1.00      1.00      2000
weighted avg       1.00      1.00      1.00      2000



In [21]:
# Test Case: Kolkata under normal conditions (e.g. 9.5 mm rain, 28°C, 80% humidity)
kolkata_normal = pd.DataFrame({
    "Rainfall (mm)": [9.5],
    "Temperature (°C)": [28.0],
    "Humidity (%)": [80.0]
})

kolkata_scaled = scaler.transform(kolkata_normal[FEATURES])
flood_idx = list(model.classes_).index(1)
prob = model.predict_proba(kolkata_scaled)[0][flood_idx]

print(f"Predicted Flood Probability for Kolkata (Normal): {prob * 100:.2f}%")

# Test Case: Heavy monsoon flood condition (180 mm rain, 26°C, 95% humidity)
kolkata_monsoon = pd.DataFrame({
    "Rainfall (mm)": [180.0],
    "Temperature (°C)": [26.0],
    "Humidity (%)": [95.0]
})
monsoon_scaled = scaler.transform(kolkata_monsoon[FEATURES])
monsoon_prob = model.predict_proba(monsoon_scaled)[0][flood_idx]

print(f"Predicted Flood Probability for Extreme Rain: {monsoon_prob * 100:.2f}%")

Predicted Flood Probability for Kolkata (Normal): 1.25%
Predicted Flood Probability for Extreme Rain: 99.70%


In [22]:
# Save artifacts so app.py picks them up directly
joblib.dump(model, "live_flood_model.pkl")
joblib.dump(scaler, "live_flood_scaler.pkl")
joblib.dump(FEATURES, "live_flood_features.pkl")

print("Artifacts successfully saved:")
print("1. live_flood_model.pkl")
print("2. live_flood_scaler.pkl")
print("3. live_flood_features.pkl")

Artifacts successfully saved:
1. live_flood_model.pkl
2. live_flood_scaler.pkl
3. live_flood_features.pkl
